# Kuiper kernel benchmarks

Each table compares the JIT-dispatched verified Kuiper kernels (`kuipy.run`) and
the unverified reference kernels (`kuipy.unverified`) against stock PyTorch, on
the shapes of a Qwen2.5-0.5B decode step.

Every GEMM here has a transposed right operand, because that is what the model
emits: `nn.Linear` stores its weight as `(N, K)` and hands ATen a view of it, so
`mm`/`addmm` see a `(K, N)` tensor with the `K` axis contiguous. That is the
layout SuperGEMM takes. The unverified references predate it and read B
row-major `(K, N)`, so they are given a copy in their own layout, made once
outside the timing loop.

Times are us/call, `rel-err` is the relative Frobenius norm against the `ref` column.

In [1]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

import kuipy
from kuipy import unverified
from kuipy.benchmarking import bench_matrix

aten = torch.ops.aten
DEV = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Qwen2.5-0.5B-Instruct, decoding at batch 256.
HID, NH, NKV, HEAD_DIM = 896, 14, 2, 64
INTER, VOCAB, BATCH = 4864, 151936, 256
# The three attention projections are fused into one addmm with a broadcast bias.
QKV = (NH + 2 * NKV) * HEAD_DIM
SCALE = HEAD_DIM ** -0.5
ALPHA, BETA = 0.75, 1.5

_g = torch.Generator(device=DEV).manual_seed(0)

def rand(*shape, dtype=torch.bfloat16):
    return torch.randn(*shape, device=DEV, dtype=dtype, generator=_g) * 0.1

torch.cuda.get_device_name(0)

'NVIDIA RTX A6000'

In [2]:
MM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("gate_proj",   BATCH, HID,   INTER),
    ("up_proj",     BATCH, HID,   INTER),
    ("down_proj",   BATCH, INTER, HID),
    ("lm_head",     BATCH, HID,   VOCAB),
    ("square_4096", 4096,  4096,  4096),
]

# B is transposed throughout: `nn.Linear` stores its weight as (N, K) and hands
# ATen a view with K contiguous, which is the layout the SuperGEMM kernels take.
def mm_inputs(dtype):
    return lambda M, K, N: ((rand(M, K, dtype=dtype),
                             rand(N, K, dtype=dtype).t()), {})

# The unverified references take B row-major (K, N); only the Kuiper kernels
# read the transposed (N, K) weight in place. Each contender therefore gets the
# layout it is written for, and the copy is cached so it is not timed.
# Keyed on the tensor object, which keeps it alive: keying on the address would
# alias, because the caching allocator hands a freed block to the next case.
_row_major_b = {}

def rm(B):
    out = _row_major_b.get(B)
    if out is None:
        out = _row_major_b[B] = B.contiguous()
    return out

# TensorCore2D.To, the non-pipelined verified kernel SuperGEMM replaces on the
# transposed-B path. It reads B row-major, so it takes the same cached copy the
# unverified contenders do; that copy is the layout advantage, not a cost.
_tc2d_mm = kuipy.run(aten.mm.default, impl="tc2d_to")
_tc2d_addmm = kuipy.run(aten.addmm.default, impl="tc2d_to")

def tc2d_to_mm(A, B):
    return _tc2d_mm(A, rm(B))

def tc2d_to(C, A, B, beta=1.0, alpha=1.0):
    return _tc2d_addmm(C, A, rm(B), beta=beta, alpha=alpha)

# gemm_tc is addmm-shaped; a bias-free matmul is just the absent epilogue term.
def gemm_tc_mm(A, B):
    return unverified.gemm_tc(None, A, rm(B), beta=0.0, alpha=1.0)

def gemm_tc(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_tc(C, A, rm(B), beta=beta, alpha=alpha)

def hacky_epilogue(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_hacky_epilogue(C, A, rm(B), beta=beta, alpha=alpha)

def gemm_pipe(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_pipe(C, A, rm(B), beta=beta, alpha=alpha)

def bcast_epilogue(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_bcast_bias_epilogue(C, A, rm(B), beta=beta, alpha=alpha)

def bcast_epilogue2(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_bcast_bias_epilogue2(C, A, rm(B), beta=beta, alpha=alpha)

MNK = lambda M, K, N: (M, K, N)
GEMM_FLOPS = lambda M, K, N: 2 * M * N * K

# A dense (M, N) epilogue term is synthetic -- the model only ever emits a
# broadcast bias -- so this is a smaller sample of the same shapes.
GEMM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
    ("square_4096", 4096,  4096,  4096),
]

# What the model actually emits: the fused qkv projection, plus the two shapes
# above for comparison against the dense-C tables.
BIAS_CASES = [
    ("qkv_proj",    BATCH, HID,   QKV),
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
]

def addmm_inputs(dtype):
    return lambda M, K, N: ((rand(M, N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(N, K, dtype=dtype).t()),
                            {"beta": BETA, "alpha": ALPHA})

_kuiper_sdpa = kuipy.run(aten._scaled_dot_product_efficient_attention.default)

def kuiper_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    return _kuiper_sdpa(q, k, v, attn_mask, False, 0.0, is_causal, scale=scale)[0]

def cudnn_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                              is_causal=is_causal, scale=scale,
                                              enable_gqa=True)

def attn_flops(sq, sk):
    return 4 * BATCH * NH * sq * sk * HEAD_DIM

DECODE_CASES = [(f"ctx_{c}", 1, c) for c in (128, 512, 1024, 16384)]

def decode_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"attn_mask": torch.zeros(BATCH, NH, sq, sk, device=DEV,
                                      dtype=torch.bfloat16),
             "scale": SCALE})

# Prefill: full self-attention, is_causal, no explicit mask.
PREFILL_CASES = [(f"seq_{s}", s, s) for s in (128, 512)]

def prefill_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"is_causal": True, "scale": SCALE})

## mm

`C = A @ B^T`. `gemm_tc` is addmm-shaped, so it appears here with its epilogue
term absent; the other unverified GEMMs are epilogue-only and show up in the
next section.

In [3]:
bench_matrix(MM_CASES, mm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.mm.default)),
              ("tc2d_to", tc2d_to_mm),
              ("gemm_tc", gemm_tc_mm)],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,tc2d_to GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,tc2d_to us,gemm_tc us,ref us,kuiper rel-err,tc2d_to rel-err,gemm_tc rel-err
0,o_proj,256,896,896,19261.420012,12032.613824,11320.022584,33788.553166,21.340160,34.160640,36.311040,12.165120,0.000000,0.000000,0.000061
1,gate_proj,256,896,4864,50278.542467,31209.853557,18935.279668,49322.588143,44.380159,71.495681,117.841921,45.240321,0.002707,0.002707,0.002707
2,up_proj,256,896,4864,51490.358026,30952.727031,18925.413095,51320.581656,43.335681,72.089601,117.903357,43.479042,0.002708,0.002708,0.002708
3,down_proj,256,4864,896,39220.158875,11567.427815,18908.989598,58170.637119,56.893439,192.901115,118.005762,38.359039,0.002623,0.002624,0.002622
4,lm_head,256,896,151936,61046.929604,56133.371874,61262.311870,91781.948955,1141.760025,1241.702423,1137.745895,759.418869,0.000000,0.000000,0.000000
5,square_4096,4096,4096,4096,63766.843325,60236.483019,66807.561873,108337.956125,2155.335693,2281.656342,2057.236481,1268.613129,0.000000,0.000000,0.000000


In [4]:
bench_matrix(MM_CASES, mm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.mm.default)),
              ("tc2d_to", tc2d_to_mm),
              ("gemm_tc", gemm_tc_mm)],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,tc2d_to GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,tc2d_to us,gemm_tc us,ref us,kuiper rel-err,tc2d_to rel-err,gemm_tc rel-err
0,o_proj,256,896,896,11397.160566,12849.167573,11982.328726,31959.235217,36.065280,31.989760,34.303999,12.861440,0.000000,0.000000,0.000029
1,gate_proj,256,896,4864,54286.798500,40548.418747,20709.675832,49932.903672,41.103358,55.029759,107.745275,44.687362,0.000338,0.000338,0.000338
2,up_proj,256,896,4864,53857.438453,40116.386175,20769.014503,49932.903672,41.431041,55.622401,107.437439,44.687362,0.000338,0.000338,0.000338
3,down_proj,256,4864,896,12747.584461,12217.267918,21617.776951,58232.815305,175.042553,182.640648,103.219204,38.318081,0.000329,0.000329,0.000329
4,lm_head,256,896,151936,51523.221692,56751.149952,77201.848678,92017.691102,1352.806396,1228.185577,902.840347,757.473297,0.000000,0.000000,0.000000
5,square_4096,4096,4096,4096,72421.718853,64593.587919,84983.431143,108488.579920,1897.758789,2127.749176,1617.244110,1266.851807,0.000000,0.000000,0.000000


## addmm

`D = beta*C + alpha*(A @ B)`: the full epilogue, which the bias-free `mm` path never hits.
`gemm_pipe` is fp16-only and `gemm_hacky_epilogue` bf16-only, hence the two tables.
`gemm_tc` handles both, and takes its epilogue term as an (M, N) matrix or a length-N
vector, so it is the only contender that appears in all three.

In [5]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("tc2d_to", tc2d_to),
              ("hacky_epilogue", hacky_epilogue),
              ("gemm_tc", gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,tc2d_to GFLOP/s,hacky_epilogue GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,tc2d_to us,hacky_epilogue us,gemm_tc us,ref us,kuiper rel-err,tc2d_to rel-err,hacky_epilogue rel-err,gemm_tc rel-err
0,o_proj,256,896,896,8959.999761,5036.526456,4889.257212,10919.695169,19755.545249,45.875201,81.612158,84.070396,37.642241,20.806401,0.000003,0.000003,0.000003,0.000041
1,down_proj,256,4864,896,10449.117481,4980.508350,4826.080678,19001.325794,51564.992119,213.546238,448.020477,462.356491,117.432318,43.272958,0.002559,0.002559,0.002559,0.002559
2,square_4096,4096,4096,4096,61844.079020,42989.018282,41500.540342,65646.264723,99469.173628,2222.346191,3197.071228,3311.738892,2093.629456,1381.724091,0.000000,0.000000,0.000000,0.000000


In [6]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("tc2d_to", tc2d_to),
              ("gemm_pipe", gemm_pipe),
              ("gemm_tc", gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,tc2d_to GFLOP/s,gemm_pipe GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,tc2d_to us,gemm_pipe us,gemm_tc us,ref us,kuiper rel-err,tc2d_to rel-err,gemm_pipe rel-err,gemm_tc rel-err
0,o_proj,256,896,896,10931.776810,5483.715953,9131.210304,11534.712790,19169.435457,37.600639,74.956799,45.015039,35.635200,21.442561,7.501406e-07,7.501406e-07,7.501406e-07,0.000024
1,down_proj,256,4864,896,12741.621331,5236.895129,10904.083577,21304.967487,51344.766844,175.124474,426.086388,204.636154,104.734716,43.458562,3.186876e-04,3.186876e-04,3.186876e-04,0.000319
2,square_4096,4096,4096,4096,70405.976838,46201.996048,66374.101827,83904.961124,98618.444457,1952.092133,2974.740601,2070.671387,1638.031311,1393.643494,0.000000e+00,0.000000e+00,0.000000e+00,0.000000


### broadcast bias

`nn.Linear` emits `addmm(bias, x, W.T)` with `bias` a length-N *vector*, not an (M, N)
matrix. A tlayout is an injection, so the stride-0 row axis of a broadcast C is
inexpressible as one; C is now read as an `rotensor` over a *virtual* tensor layout,
which need not be injective, and the out-of-place epilogue reads C and writes D through
independent index functions, so dropping C's row term costs one term in the index
expression. `kuiper` is that verified path. `gemm_bcast_bias_epilogue` and
`gemm_bcast_bias_epilogue2` are the hand-edited extractions that prototyped it (the
former stages the bias slice into shared memory replicated over a fragment's rows, the
latter is the one-line lift of `TensorCore2D.To`). `kuiper+materialise` is what the
verified path had to do before: materialise the full (M, N) C and run the stock kernel.

In [7]:
def bias_inputs(dtype):
    return lambda M, K, N: ((rand(N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(N, K, dtype=dtype).t()),
                            {"beta": BETA, "alpha": ALPHA})

_kuiper_addmm = kuipy.run(aten.addmm.default)

def kuiper_dense_bias(bias, A, B, beta=1.0, alpha=1.0):
    return _kuiper_addmm(bias.expand(A.size(0), B.size(1)).contiguous(), A, B,
                         beta=beta, alpha=alpha)

bench_matrix(BIAS_CASES, bias_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", _kuiper_addmm),
              ("kuiper+materialise", kuiper_dense_bias),
              ("bcast_epilogue", bcast_epilogue),
              ("bcast_epilogue2", bcast_epilogue2),
              ("gemm_tc", gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,kuiper+materialise GFLOP/s,bcast_epilogue GFLOP/s,bcast_epilogue2 GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,...,kuiper+materialise us,bcast_epilogue us,bcast_epilogue2 us,gemm_tc us,ref us,kuiper rel-err,kuiper+materialise rel-err,bcast_epilogue rel-err,bcast_epilogue2 rel-err,gemm_tc rel-err
0,qkv_proj,256,896,1152,14686.852666,11576.850730,7042.794837,7150.124675,11745.471283,23416.999186,...,45.649920,75.038719,73.912320,44.994559,22.568319,0.000002,0.000002,0.000000,0.000002,0.000023
1,o_proj,256,896,896,11384.231187,9004.217234,5480.767695,5555.051147,12141.802681,17697.789477,...,45.649920,74.997120,73.994241,33.853440,23.225601,0.000003,0.000003,0.000000,0.000003,0.000025
2,down_proj,256,4864,896,12998.520594,12601.618988,5381.222085,5402.033034,21280.000538,50961.267957,...,177.070084,414.658546,413.061104,104.857597,43.785601,0.000319,0.000319,0.000319,0.000319,0.000319


## sdpa

The decode mask is a dense `(B, Hq, Sq, Sk)` tensor: the Kuiper mask is read through a
`rotensor`, whose layout need not be an injection, but the instantiation template only
emits the dense row-major layout, so the mask is materialised and handed to every
contender for fairness. Prefill passes no mask at all, which the kernel selects with a
broadcast layout plus `has_mask = false`.

In [8]:
bench_matrix(DECODE_CASES, decode_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("manual_extract", unverified.flash_attn_manual_extract),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,manual_extract GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,manual_extract us,fa1 us,fa2 us,ref us,kuiper rel-err,manual_extract rel-err,fa1 rel-err,fa2 rel-err
0,ctx_128,1,128,738.113034,756.517153,826.997383,2271.049593,2386.350491,159.109116,155.238400,142.008324,51.711998,49.213438,0.002327,0.002327,0.002327,0.001661
1,ctx_512,1,512,1052.521433,1047.905376,1029.284295,2999.555465,3789.459632,446.320648,448.286705,456.396790,156.610556,123.965445,0.002350,0.002350,0.002350,0.001620
2,ctx_1024,1,1024,1199.351656,1201.424733,1127.016361,2793.180666,3907.951686,783.359985,782.008286,833.638382,336.363525,240.413437,0.002360,0.002360,0.002360,0.001465
3,ctx_16384,1,16384,1536.632358,1538.819817,1393.781926,3955.163542,4036.599648,9782.681885,9768.775635,10785.321045,3800.698853,3724.021912,0.002323,0.002323,0.002323,0.000878


In [9]:
bench_matrix(PREFILL_CASES, prefill_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,fa1 us,fa2 us,ref us,kuiper rel-err,fa1 rel-err,fa2 rel-err
0,seq_128,128,128,3503.896290,5203.150416,14455.141550,64761.177867,4290.191345,2889.093018,1039.933472,232.120323,0.001299,0.000927,0.000927
1,seq_512,512,512,7391.951008,9564.386529,35254.080738,112141.807939,32537.846680,25147.265625,6822.420654,2144.768066,0.001560,0.001074,0.001074
